In [1]:
%%capture
!pip install -U sagemaker

## Dependencies

In [3]:
import boto3
import pandas as pd
import numpy as np

from sagemaker import get_execution_role
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder

/opt/conda/lib/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


## Setup

In [4]:
sm_boto3 = boto3.client("sagemaker")
sess = sagemaker.Session()
region = sess.boto_session.region_name
bucket = "sklearn-mlops-tutorial"
print("Using bucket " + bucket)

Using bucket sklearn-mlops-tutorial


In [5]:
df = pd.read_csv("./data/mob_price_classification_train.csv")

## Process data

In [6]:
features = list(df.columns)
label = features.pop(-1)

In [7]:
x = df[features]
y = df[label]

#### Split Data

In [8]:
X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.15, random_state=0)

In [9]:
trainX = pd.DataFrame(X_train)
trainX[label] = y_train

testX = pd.DataFrame(X_test)
testX[label] = y_test

In [10]:
trainX.to_csv("./processed_data/train-V-1.csv",index = False)
testX.to_csv("./processed_data/test-V-1.csv", index = False)

#### Upload data to S3

In [11]:
# send data to S3. SageMaker will take training data from s3
sk_prefix = "sagemaker/mobile_price_classification/sklearncontainer"

trainpath = sess.upload_data(
    path="./processed_data/train-V-1.csv", bucket=bucket, key_prefix=sk_prefix
)

testpath = sess.upload_data(
    path="./processed_data/test-V-1.csv", bucket=bucket, key_prefix=sk_prefix
)

In [12]:
# # test the script locally
# ! python code/script.py --n_estimators 100 \
#                    --random_state 0 \
#                    --model-dir ./ \
#                    --train ./ \
#                    --test ./ \


## Deploy training job using SKLearn container

In [13]:
FRAMEWORK_VERSION = "0.23-1"

sklearn_estimator = SKLearn(
    source_dir="code",
    entry_point="script.py",
    role=get_execution_role(),
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version=FRAMEWORK_VERSION,
    base_job_name="RF-custom-sklearn",
    hyperparameters={
        "n_estimators": 100,
        "random_state": 0,
    },
    use_spot_instances = True,
    max_wait = 7200,
    max_run = 3600
)

In [14]:
sklearn_estimator.fit({"train": trainpath, "test": testpath}, wait=True)

[12/27/24 17:13:39] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=103179;file:///opt/conda/lib/python3.11/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=373640;file:///opt/conda/lib/python3.11/site-packages/sagemaker/telemetry/telemetry_logging.py#90\90]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating training-job with name:                                       ]8;id=345771;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=208292;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py#1042\1042]8;;\
                             RF-custom-sklearn-2024-12-27-17-13-39-444                                             

2024-12-27 17:13:41 Starting - Starting the training job...
2024-12-27 17:13:54 Starting - Preparing the instances for training...
2024-12-27 17:14:20 Downloading - Downloading input data...
2024-12-27 17:14:45 Downloading - Downloading the training image...
2024-12-27 17:15:36 Training - Training image download completed. Training in progress.
2024-12-27 17:15:36 Uploading - Uploading generated training model2024-12-27 17:15:29,992 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2024-12-27 17:15:29,996 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2024-12-27 17:15:30,041 sagemaker_sklearn_container.training INFO     Invoking user training script.
2024-12-27 17:15:30,189 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2024-12-27 17:15:30,202 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2024-12-27 17:15:30,214 sagemaker-training-toolkit INFO

## Store Artifacts in S3 Bucket

In [21]:
sklearn_estimator.latest_training_job.wait(logs="None")
artifact = sm_boto3.describe_training_job(TrainingJobName=sklearn_estimator.latest_training_job.name)["ModelArtifacts"]["S3ModelArtifacts"]

print("Model artifact persisted at " + artifact)


2024-12-27 09:51:32 Starting - Preparing the instances for training
2024-12-27 09:51:32 Downloading - Downloading the training image
2024-12-27 09:51:32 Training - Training image download completed. Training in progress.
2024-12-27 09:51:32 Uploading - Uploading generated training model
2024-12-27 09:51:32 Completed - Training job completed
Model artifact persisted at s3://sagemaker-us-east-1-875716602731/RF-custom-sklearn-2024-12-27-09-49-12-814/output/model.tar.gz
